# 12 · Prognosis Scaling

Scale a neighbourhood's population to a future year (2025–2032) using the official Gothenburg population prognosis, and compare demographic shifts.

In [1]:
import logging
logging.basicConfig(level=logging.ERROR)

from gbgsynth import GbgSynth

city = GbgSynth(year=2023)

### Preview scaling factors

In [2]:
preview = city.get_area("Haga").get_prognosis_summary(
    base_year=2025, target_year=2030
)
print("Prognosis preview (Haga, 2025→2030):")
for k, v in preview.items():
    print(f"  {k}: {v}")

Prognosis preview (Haga, 2025→2030):
  pri_code: 107
  mel_code: 34
  mel_name: Olivedal-Haga-Annedal-Änggården
  base_year: 2025
  target_year: 2030
  base_population: 20556
  target_population: 20595
  overall_growth: +0.2%
  scale_factors: {'0-17 år': np.float64(0.9066), '18-24 år': np.float64(1.0136), '25-44 år': np.float64(1.0053), '45-64 år': np.float64(0.9722), '65-79 år': np.float64(0.9932), '80+ år': np.float64(1.329), '_overall': np.float64(1.0019)}


### Generate base and future populations

In [3]:
base = city.synthesize("Haga")
future = city.synthesize_future("Haga", target_year=2030)

### Compare demographic shifts

In [4]:
from collections import Counter

def age_bucket(age):
    if age < 18: return "0-17"
    if age < 65: return "18-64"
    return "65+"

base_ages = Counter(age_bucket(a.age) for a in base.individuals)
future_ages = Counter(age_bucket(a.age) for a in future.individuals)

print(f"{'Age group':<12} {'2023':>8} {'2030':>8} {'Change':>8}")
print("-" * 38)
for grp in ["0-17", "18-64", "65+"]:
    b, f = base_ages[grp], future_ages[grp]
    pct = (f - b) / b * 100 if b else 0
    print(f"{grp:<12} {b:>8,} {f:>8,} {pct:>+7.1f}%")

print(f"{'Total':<12} {len(base.individuals):>8,} "
      f"{len(future.individuals):>8,} "
      f"{(len(future.individuals) - len(base.individuals)) / len(base.individuals) * 100:>+7.1f}%")

Age group        2023     2030   Change
--------------------------------------
0-17              561      497   -11.4%
18-64           2,330    2,306    -1.0%
65+               917      964    +5.1%
Total           3,808    3,767    -1.1%
